# Day 052 — Exercise 5: Assemble the Full AI API

**What you'll build:** `build_api(model)` — one FastAPI app that combines everything: `GET /health`, `GET /templates`, `POST /chat`, and `POST /render/{name}` (a templated chat call). This is the deliverable you'll ship as `main.py`.

**Why it matters:** A real service is many routes on one app, sharing models and helpers. `build_api` is the factory that assembles them. The provided `run_model` helper keeps each route thin — the same thin-handler-over-logic pattern you used for the Streamlit `ChatApp`, now on the server side.

## Provided: Setup + Models + Templates + run_model helper

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from starlette.testclient import TestClient
import ollama


class ChatRequest(BaseModel):
    """Request body for the chat endpoints."""
    message: str = Field(min_length=1, description='User message for the model')
    temperature: float = Field(default=0.7, ge=0.0, le=1.0)


class ChatResponse(BaseModel):
    """Response body returned by the chat endpoints."""
    reply: str
    model: str


class HealthResponse(BaseModel):
    """Response body for the health check."""
    status: str
    model: str


PROMPT_TEMPLATES = {
    'summary':  'Summarize the following topic in two sentences: {topic}',
    'explain':  'Explain {topic} to a complete beginner.',
    'critique': 'List three criticisms of {topic}.',
}


def run_model(model: str, prompt: str, temperature: float = 0.7) -> str:
    """Call Ollama once and return the reply text. Raises on model error."""
    resp = ollama.chat(
        model=model,
        messages=[{'role': 'user', 'content': prompt}],
        options={'temperature': temperature},
    )
    return resp['message']['content'].strip()

## Your Implementation

In [ ]:
def build_api(model: str = 'llama3.2') -> FastAPI:
    """
    Assemble the full AI API:
      GET  /health            -> HealthResponse(status='ok', model=model)
      GET  /templates         -> {'templates': [names]}
      POST /chat              -> ChatResponse from run_model(model, req.message, ...)
      POST /render/{name}     -> ChatResponse from the named template (404 if unknown)
    """
    app = FastAPI(title='AI API', version='1.0.0')

    # TODO: @app.get('/health', response_model=HealthResponse)
    #       def health(): return HealthResponse(status='ok', model=model)

    # TODO: @app.get('/templates')
    #       def list_templates(): return {'templates': list(PROMPT_TEMPLATES.keys())}

    # TODO: @app.post('/chat', response_model=ChatResponse)
    #       def chat(req: ChatRequest):
    #           try: return ChatResponse(reply=run_model(model, req.message, req.temperature), model=model)
    #           except Exception as e: raise HTTPException(status_code=503, detail=f'Model unavailable: {e}')

    # TODO: @app.post('/render/{name}', response_model=ChatResponse)
    #       def render_chat(name: str, req: ChatRequest):
    #           if name not in PROMPT_TEMPLATES: raise HTTPException(status_code=404, detail=...)
    #           prompt = PROMPT_TEMPLATES[name].format(topic=req.message)
    #           try: return ChatResponse(reply=run_model(model, prompt, req.temperature), model=model)
    #           except Exception as e: raise HTTPException(status_code=503, detail=f'Model unavailable: {e}')

    return app

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    # Check 1: build_api returns an app exposing all four routes
    try:
        app = build_api()
        assert isinstance(app, FastAPI)
        paths = {r.path for r in app.routes}
        for p in ('/health', '/templates', '/chat', '/render/{name}'):
            assert p in paths, f'missing route: {p} (have {sorted(paths)})'
        client = TestClient(app)
        passed += 1; print('✅ Check 1: build_api exposes health/templates/chat/render')
    except Exception as e:
        print(f'❌ Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: GET /health -> 200 with status + model
    try:
        b = client.get('/health').json()
        assert b['status'] == 'ok' and b['model'] == 'llama3.2', f'bad health: {b}'
        passed += 1; print('✅ Check 2: /health reports ok + model')
    except Exception as e:
        print(f'❌ Check 2: {e}')

    # Check 3: GET /templates lists the names
    try:
        assert 'summary' in client.get('/templates').json()['templates']
        passed += 1; print('✅ Check 3: /templates lists names')
    except Exception as e:
        print(f'❌ Check 3: {e}')

    # Check 4: POST /chat -> 200 with a real reply (Ollama)
    try:
        r = client.post('/chat', json={'message': 'Reply with exactly: pong'})
        assert r.status_code == 200, f'expected 200, got {r.status_code} ({r.text[:120]})'
        assert len(r.json()['reply']) > 0
        passed += 1; print('✅ Check 4: POST /chat returns a reply')
    except Exception as e:
        print(f'❌ Check 4: {e}')

    # Check 5: POST /render/{name} — valid renders (200), unknown -> 404
    try:
        ok = client.post('/render/summary', json={'message': 'Python'})
        assert ok.status_code == 200, f'expected 200, got {ok.status_code} ({ok.text[:120]})'
        assert len(ok.json()['reply']) > 0
        missing = client.post('/render/nope', json={'message': 'x'})
        assert missing.status_code == 404, f'expected 404, got {missing.status_code}'
        passed += 1; print('✅ Check 5: /render works (200) and 404s on unknown template')
    except Exception as e:
        print(f'❌ Check 5: {e}')

    if passed == total:
        print('🎉 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def build_api(model: str = 'llama3.2') -> FastAPI:
    """Assemble the complete AI API: health, templates, chat, and templated chat."""
    app = FastAPI(title='AI API', version='1.0.0')

    @app.get('/health', response_model=HealthResponse)
    def health():
        return HealthResponse(status='ok', model=model)

    @app.get('/templates')
    def list_templates():
        return {'templates': list(PROMPT_TEMPLATES.keys())}

    @app.post('/chat', response_model=ChatResponse)
    def chat(req: ChatRequest):
        try:
            return ChatResponse(reply=run_model(model, req.message, req.temperature),
                                model=model)
        except Exception as e:
            raise HTTPException(status_code=503, detail=f'Model unavailable: {e}')

    @app.post('/render/{name}', response_model=ChatResponse)
    def render_chat(name: str, req: ChatRequest):
        if name not in PROMPT_TEMPLATES:
            raise HTTPException(status_code=404, detail=f'template {name!r} not found')
        prompt = PROMPT_TEMPLATES[name].format(topic=req.message)
        try:
            return ChatResponse(reply=run_model(model, prompt, req.temperature),
                                model=model)
        except Exception as e:
            raise HTTPException(status_code=503, detail=f'Model unavailable: {e}')

    return app
```

**Why this works:** `build_api` registers four routes on one app, all sharing the `ChatRequest`/`ChatResponse` contract and the `run_model` helper. `/render/{name}` combines everything from the day: a path parameter, a request body, a 404 for unknown templates, a model call, and a 503 on failure. Returning the app from a factory (rather than a global) means the tests can build fresh instances — and `main.py` just calls `app = build_api()` for uvicorn to serve.
</details>